# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. See [FAIR² Croissant Schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for metadata.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and explore the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata (as a Python object)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
We'll review available record sets, their `@id`'s, fields, and columns. All entities are referenced by their Croissant `@id`.

In [ ]:
# Print all available record sets and their fields by @id

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields (columns/@id):")
    for field in rs.fields:
        # field.id returns the field's @id
        print(f"    - {field.name} (@id: {field.id}) (type: {field.data_type})")
    print()

## 3. Data Extraction
Let's extract a selected record set into a DataFrame for analysis. Use record set and field `@id` values as shown above for reference.

**Note:** This dataset contains one main record set describing the protein data table; here we extract it by its `@id`.

In [ ]:
# For this dataset, the main record set contains protein entries. We'll extract its @id automatically:

# Typically there is one table
main_record_set = dataset.record_sets[0]
main_record_set_id = main_record_set.id

# List all record set ids (in general there may be more than one)
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    # Load all records/rows for this record set
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

df = dataframes[main_record_set_id]
print(f"DataFrame columns for record set {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We'll explore the data: filtering proteins by a numeric field such as 'molecular_weight' or 'peptide_count', and perform normalization/groupings.

_Replace `molecular_weight_field_id` and `group_field_id` below with the appropriate field `@id` references from the Data Overview._

In [ ]:
# Choose a numeric field. We'll auto-detect one that's available.

import numpy as np

# Print all fields again for reference
for field in main_record_set.fields:
    print(f"Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")

# Let's try commonly expected numeric fields
possible_numeric_fields = ['molecular_weight', 'MW', 'MW_(kDa)', 'molecular_weight_kda', 'peptide_count', 'Peptides', 'peptide_count_per_protein']
numeric_field_id = None
for col in df.columns:
    for guess in possible_numeric_fields:
        if guess.lower() in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        break

if numeric_field_id is None:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]
    else:
        raise RuntimeError("No numeric column found in data!")

print(f"Selected numeric field for EDA: '{numeric_field_id}'")

# Try to clean/convert column to float if not already
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter rows: proteins with molecular_weight > threshold
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (median): {len(filtered_df)} records")

# Normalize the numeric column
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, numeric_field_id+'_normalized']].head())

# Try grouping by a categorical field, e.g. 'description' or 'modification'
group_field_id = None
for field in main_record_set.fields:
    if field.data_type == 'Text' and field.id != numeric_field_id:
        group_field_id = field.id if field.id in df.columns else field.name if field.name in df.columns else None
        if group_field_id:
            break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped by '{group_field_id}', average of {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of protein molecular weight (or the detected numeric field) and relationships with a categorical variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

if group_field_id is not None:
    # Top categories
    top_groups = df[group_field_id].value_counts().head(5).index.tolist()
    filtered_plot_df = df[df[group_field_id].isin(top_groups)]
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_plot_df)
    plt.title(f"{numeric_field_id} by {group_field_id} (Top 5 groups)")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook demonstrates how to use the `mlcroissant` library to load and explore a multi-field protein mass spectrometry Croissant dataset. We loaded data directly via Croissant schema, inspected available record sets, referenced fields and record sets via their `@id`s, performed EDA and filtering, and visualized the distribution of a key numeric field. This approach is extensible to other FAIR² and Croissant-structured datasets.

_Further analysis (e.g. advanced outlier removal, PCA, clustering) can be performed using standard pandas, scikit-learn, or other data science libraries once the data is extracted as above._